# design_04 집중 재추출 · 백본 1개 × 서열 50개 · v0.3.0

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_protein/blob/main/notebooks/04_focus_design04_T4.ipynb?v=0.3.0)

**백본 하나를 고정하고 서열만 50개 새로 뽑아 같은 필터로 채점합니다.** RFdiffusion3 백본 생성 단계가 없으므로 **약 30분**이면 끝납니다(설치 포함).

| 단계 | 도구 | 산출 |
| --- | --- | --- |
| 1 | — | 저장소에 있는 백본 `design_04` 복사 (생성 안 함) |
| 2 | LigandMPNN | 서열 50개 (seed 1–5 × 배치 10) |
| 3 | AF2 initial guess | 설계 50개 + 양성 대조군 1개 채점 |

## 무엇을 가르려는 실행인가

[설계 160개 실행](https://github.com/HisameOgasahara/tmp_protein/blob/main/docs/04-scale-run.md)에서 `design_04` 백본이 특이했습니다.

| 항목 | 값 |
| --- | --- |
| 표적에 대한 배치 (`target_aligned_rmsd`) | **1.190 Å** — 공개 결합여 TrkA_88(1.497 Å)보다 정확 |
| 서열 8개 중 접힘·배치를 동시에 만족 | **7개** |
| 그런데 `pae_interaction` 최저 | **12.934** — 통과선 10 밖 |

자리는 맞는데 AF2가 그 계면을 믿지 않습니다. 해석이 둘이고, 다음에 계산 시간을 어디 쓸지가 여기서 갈립니다.

| 해석 | 이 실행에서 보이는 모습 | 그러면 다음은 |
| --- | --- | --- |
| **백본의 한계** | 서열 50개 전부가 10을 못 넘는다 | 모양을 더 많이 만들어야 한다 |
| **서열 탐색의 한계** | 하나라도 10 아래로 내려간다 | 좋은 모양에 서열을 더 쏟아야 한다 |

8개로는 구분되지 않습니다. 같은 백본에 50개를 뽑으면 구분됩니다.

## 왜 짧은가

백본을 새로 만들지 않고 [v0.2.0 실행 결과](https://github.com/HisameOgasahara/tmp_protein/tree/main/results/2026-11-19-trka-design-v2/backbones)에 커밋된 `design_04.pdb`를 그대로 씁니다. 같은 구조이므로 앞 실행의 서열 8개와 직접 비교됩니다.

AF2에 들어가는 길이도 2종뿐입니다 — 설계 73잔기, 대조군 93잔기. **JAX 컴파일이 2회**로 끝납니다(길이당 약 92초).

## 이번에 처음 기록하는 것

**MPNN이 매긴 서열별 점수**입니다. 지금까지는 서열 문자열만 저장하고 설계기 자신의 점수는 버렸습니다. 이 점수가 `pae_interaction`과 상관이 있으면 **AF2에 보내기 전에 서열을 고를 수 있다**는 뜻이고, 그게 다음 규모 확대에 남은 절약 수단입니다.

필드 이름을 추측하지 않고 결과 객체가 들고 있는 스칼라를 전부 긁어 기록합니다. 이름을 잘못 짚으면 조용히 아무것도 안 남기 때문입니다.

## 통과 기준

논문 기준 그대로 `pae_interaction < 10`, `plddt_binder > 80`, `binder_aligned_rmsd < 1 Å`입니다. 통과가 합격을 뜻하지 않는 이유는 [교정 결과](https://github.com/HisameOgasahara/tmp_protein/blob/main/docs/02-filter-calibration.md)에 있습니다.

[코드](https://github.com/HisameOgasahara/tmp_protein/blob/main/src/rfd3_design/focus_design.py)


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import subprocess
import sys

checkout = Path(mkdtemp(prefix="trka_focus_main_", dir="/content")) / "repo"
clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--single-branch", "--branch", "main",
     "https://github.com/HisameOgasahara/tmp_protein.git", str(checkout)],
    capture_output=True, text=True,
)
if clone.returncode:
    print(clone.stderr)
    clone.check_returncode()
revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
print(f"main: {revision[:12]}")


In [ ]:
import json
import pandas as pd
from IPython.display import display

PASS_PAE = 10.0

log = checkout.parent / "trka_focus.log"
result_zip = None
with log.open("w", encoding="utf-8") as handle:
    with subprocess.Popen(
        [sys.executable, "-u", str(checkout / "src/rfd3_design/focus_workflow.py"), revision],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
    ) as process:
        for line in process.stdout:
            handle.write(line)
            handle.flush()
            print(line, end="", flush=True)
            if line.strip().endswith(".zip"):
                result_zip = Path(line.strip())
        if process.wait():
            raise RuntimeError(f"Focus run failed. Full log: {log}")
if result_zip is None or not result_zip.is_file():
    raise FileNotFoundError(f"Missing result ZIP. Full log: {log}")

folder = result_zip.with_suffix("")
table = folder / "designs.csv"
if not table.is_file():
    print("설계 또는 채점 단계 오류 기록이 ZIP에 저장됐습니다.")
else:
    frame = pd.read_csv(table)
    new = frame[(~frame.control) & (frame.status == "success")]

    print()
    print("=== 이번 50개 (결합면 PAE 순, 상위 15개) ===")
    columns = ["name", "passes_filter", "pae_interaction", "plddt_binder",
               "binder_aligned_rmsd", "target_aligned_rmsd"]
    display(new.reindex(columns=columns).sort_values("pae_interaction").head(15).round(3))

    # 같은 백본에 앞 실행이 뽑았던 서열 8개. 비교 기준입니다.
    rows = []
    previous = checkout / "results/2026-11-19-trka-design-v2/designs.csv"
    if previous.is_file():
        old = pd.read_csv(previous)
        old = old[old.name.str.startswith("design_04_seq")]
        rows.append({"집단": f"v0.2.0 서열 {len(old)}개",
                     "PAE 최소": old.pae_interaction.min(),
                     "PAE 중앙": old.pae_interaction.median(),
                     "PAE 최대": old.pae_interaction.max(),
                     "10 미만": int((old.pae_interaction < PASS_PAE).sum()),
                     "세 기준 통과": int(old.passes_filter.sum())})
    rows.append({"집단": f"이번 서열 {len(new)}개",
                 "PAE 최소": new.pae_interaction.min(),
                 "PAE 중앙": new.pae_interaction.median(),
                 "PAE 최대": new.pae_interaction.max(),
                 "10 미만": int((new.pae_interaction < PASS_PAE).sum()),
                 "세 기준 통과": int(new.passes_filter.sum())})
    print()
    print("=== 같은 백본, 서열만 다르게 ===")
    display(pd.DataFrame(rows).round(3))

    crossed = new[new.pae_interaction < PASS_PAE]
    print()
    print("=== 판정 ===")
    if len(crossed):
        print(f"서열 탐색의 한계였습니다. 50개 중 {len(crossed)}개가 PAE {PASS_PAE} 아래로 "
              f"내려갔습니다 (최저 {crossed.pae_interaction.min():.3f}).")
        print("-> 좋은 백본에 서열을 더 뽑는 쪽에 계산 시간을 씁니다.")
    else:
        print(f"50개 전부 PAE {PASS_PAE} 위입니다 (최저 {new.pae_interaction.min():.3f}).")
        print("-> 이 백본의 한계로 읽힙니다. 백본을 더 많이 만드는 쪽에 계산 시간을 씁니다.")

    # MPNN 점수로 AF2에 보낼 서열을 미리 고를 수 있는지.
    scores = folder / "mpnn_scores.json"
    if scores.is_file():
        designs = json.loads(scores.read_text(encoding="utf-8"))
        wide = pd.DataFrame([{"name": d["name"], "mpnn_seed": d["mpnn_seed"],
                              **d.get("mpnn", {})} for d in designs])
        merged = wide.merge(new[["name", "pae_interaction", "plddt_binder"]], on="name")
        numeric = [c for c in wide.columns if c != "name"
                   and pd.api.types.is_numeric_dtype(merged[c])
                   and merged[c].nunique() > 1]
        print()
        print("=== MPNN 점수가 AF2 결과를 미리 가르는가 ===")
        if numeric:
            display(pd.DataFrame([{"MPNN 값": c,
                                   "corr(pae)": merged[c].corr(merged.pae_interaction),
                                   "corr(plddt)": merged[c].corr(merged.plddt_binder)}
                                  for c in numeric]).round(3))
            print("상관이 0 근처면 MPNN 점수로는 AF2에 보낼 서열을 미리 고를 수 없습니다.")
        else:
            print("기록된 스칼라가 없습니다. runtime/design_sequences.log에서 "
                  "MPNN 결과 객체가 무엇을 들고 있는지 확인하세요.")

    summary = folder / "designs_summary.json"
    if summary.is_file():
        print()
        print(json.dumps(json.loads(summary.read_text(encoding="utf-8")),
                         indent=2, ensure_ascii=False))
print(result_zip)


In [ ]:
from google.colab import files
files.download(str(result_zip))
